In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

        self.out = nn.Linear(d_model, d_model)

    def _rotate_half(self, x):
        # x: (batch, num_heads, seq_len, d_k)
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    def _rope(self, q, k, base=10000):
        # q, k: (batch, num_heads, seq_len, d_k)
        seq_len = q.size(-2)
        freqs = torch.arange(0, self.d_k, 2, device=q.device) / self.d_k
        freqs = base ** -freqs
        angles = torch.arange(seq_len, device=q.device)[:, None] * freqs[None, :]
        sin = angles.sin().repeat_interleave(2, dim=-1)[None, None]
        cos = angles.cos().repeat_interleave(2, dim=-1)[None, None]
        return q * cos + self._rotate_half(q) * sin, \
              k * cos + self._rotate_half(k) * sin

    def forward(self, x, attn_mask=None):
        """
        x: (batch, seq_len, d_model)
        """
        batch_size, seq_len, _ = x.shape

        Q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        Q, K = self._rope(Q, K)
        score = Q @ K.transpose(-2, -1) / self.d_k ** 0.5
        
        # 新增的因果掩码，确保每个位置只能关注之前的位置
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),
            diagonal=1
        )
        score = score.masked_fill(causal_mask[None, None, :, :], -1e9)

        if attn_mask is not None:
            key_mask = attn_mask[:, None, None, :]
            score = score.masked_fill(key_mask == 0, -1e9)

        attn_w = torch.softmax(score, dim=-1)
        
        output = self.dropout(attn_w) @ V
        output = output.transpose(1, 2).reshape(batch_size, -1, self.d_model)
        output = self.out(output)
        return output


class TransformerDecoderBlock(nn.Module): # 修改block名称即可
    def __init__(self, ffn_dim, d_model, num_heads, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, attn_mask=None):
        x_norm = self.norm1(x)
        x = x + self.dropout(self.attn(x_norm, attn_mask=attn_mask))
        x_norm = self.norm2(x)
        x = x + self.dropout(self.ffn(x_norm))
        return x


class TransformerDecoder(nn.Module): # 修改encoder名称即可
    def __init__(self, vocab_size, output_dim, ffn_dim, d_model, num_heads, num_layers, dropout=0.1, padding_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(ffn_dim, d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.out = nn.Linear(d_model, output_dim)

    def forward(self, x, attn_mask=None):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x, attn_mask=attn_mask)
        return self.out(x)